# 01 — 1 fold 全サイズ学習

`FOLD` を指定して、`SIZES` のすべてのサイズを順次実行する。
結果はDriveに保存される（既存の結果はスキップ）。

5 fold × 6 size = 30ジョブを **5回の実行（fold単位）** でまとめて処理できる。

**GPUランタイム推奨。**

In [ ]:
# ===== ここを変えて実行する =====
FOLD  = 1                           # 1〜5
SIZES = [15, 30, 60, 90, 120, 149] # 実行するサイズ一覧（既存結果はスキップ）
# ================================

DRIVE_DATA_DIR = "/content/drive/MyDrive/anglist_data"
DRIVE_LC_DIR   = "/content/drive/MyDrive/anglist_learning_curve"
EPOCHS     = 100
BATCH_SIZE = 4
RESIZE     = [512, 512]
SIGMA      = 4.0
LR         = 1e-3

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!git clone https://github.com/masaki39/anglist /content/anglist
!pip install -q tqdm onnxruntime onnxscript

In [ ]:
import json, math, os, random, shutil, subprocess, sys
import numpy as np
import onnx
import onnxruntime as ort
import torch

print("CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

sys.path.insert(0, "/content/anglist/train")
from dataset import LANDMARK_ORDER, _percentile_clip_norm, _resize_with_padding

# フォールド割り当てを読み込む
with open(os.path.join(DRIVE_LC_DIR, "folds.json")) as f:
    fold_data = json.load(f)

test_ids       = fold_data["folds"][str(FOLD)]
all_train_ids  = [cid for k, ids in fold_data["folds"].items() for cid in ids if k != str(FOLD)]

# テストデータをローカルにコピー（全サイズ共通）
LOCAL_TEST = "/content/test_data"
if os.path.exists(LOCAL_TEST): shutil.rmtree(LOCAL_TEST)
os.makedirs(LOCAL_TEST)
for cid in test_ids:
    for ext in ["_image.npy", "_landmarks.json"]:
        shutil.copy(os.path.join(DRIVE_DATA_DIR, cid + ext), LOCAL_TEST)

print(f"Fold {FOLD} | test={len(test_ids)} | sizes={SIZES}")

def preprocess(img_np, resize=(512, 512)):
    if img_np.ndim == 3: img_np = img_np[0]
    img_np = _percentile_clip_norm(img_np)
    t = torch.from_numpy(img_np).unsqueeze(0)
    t, scale, pad_x, pad_y = _resize_with_padding(t, resize)
    return t.unsqueeze(0), scale, pad_x, pad_y

def postprocess(hm):
    hm = hm[0]
    return [(float(np.unravel_index(np.argmax(c), c.shape)[1]),
             float(np.unravel_index(np.argmax(c), c.shape)[0])) for c in hm]

results_dir = os.path.join(DRIVE_LC_DIR, "results")
os.makedirs(results_dir, exist_ok=True)

for SIZE in SIZES:
    out_path = os.path.join(results_dir, f"fold{FOLD}_size{SIZE:03d}.json")
    if os.path.exists(out_path):
        print(f"[skip] fold{FOLD}_size{SIZE:03d} already exists")
        continue

    print(f"\n{'='*50}")
    print(f"FOLD={FOLD}  SIZE={SIZE}")
    print(f"{'='*50}")

    # --- データ準備 ---
    random.seed(FOLD * 1000 + SIZE)
    train_ids = sorted(all_train_ids)
    random.shuffle(train_ids)
    train_ids = train_ids[:SIZE]

    LOCAL_TRAIN = "/content/train_data"
    if os.path.exists(LOCAL_TRAIN): shutil.rmtree(LOCAL_TRAIN)
    os.makedirs(LOCAL_TRAIN)
    for cid in train_ids:
        for ext in ["_image.npy", "_landmarks.json"]:
            shutil.copy(os.path.join(DRIVE_DATA_DIR, cid + ext), LOCAL_TRAIN)
    print(f"train={len(train_ids)}")

    # --- 学習 ---
    SAVE_DIR = f"/content/runs/fold{FOLD}_size{SIZE:03d}"
    subprocess.run([
        "python", "train.py",
        "--data-dir", LOCAL_TRAIN,
        "--save-dir", SAVE_DIR,
        "--epochs", str(EPOCHS),
        "--batch-size", str(BATCH_SIZE),
        "--lr", str(LR),
        "--sigma", str(SIGMA),
        "--resize", str(RESIZE[0]), str(RESIZE[1]),
    ], cwd="/content/anglist/train", check=True)

    # --- ONNX エクスポート ---
    subprocess.run([
        "python", "export_onnx.py",
        "--checkpoint", f"{SAVE_DIR}/best.pt",
        "--output", "/content/model_raw.onnx",
        "--height", str(RESIZE[0]),
        "--width", str(RESIZE[1]),
    ], cwd="/content/anglist/train", check=True)

    proto = onnx.load("/content/model_raw.onnx", load_external_data=True)
    onnx.save_model(proto, "/content/model.onnx", save_as_external_data=False)

    # --- 評価 ---
    sess = ort.InferenceSession("/content/model.onnx", providers=["CPUExecutionProvider"])
    all_errors = {name: [] for name in LANDMARK_ORDER}

    for cid in test_ids:
        img_np = np.load(os.path.join(LOCAL_TEST, cid + "_image.npy"))
        inp_t, scale, pad_x, pad_y = preprocess(img_np)
        ort_out = sess.run(None, {"image": inp_t.numpy()})
        pred = postprocess(ort_out[0])

        with open(os.path.join(LOCAL_TEST, cid + "_landmarks.json")) as f:
            meta = json.load(f)
        spacing = meta["metadata"]["spacing"][0]
        lm = meta["landmarks_ijk"]

        for (px, py), name in zip(pred, LANDMARK_ORDER):
            gx, gy = lm[name]["i"], lm[name]["j"]
            x_o = (px - pad_x) / scale
            y_o = (py - pad_y) / scale
            all_errors[name].append(math.sqrt((x_o - gx)**2 + (y_o - gy)**2) * spacing)

    # --- 集計・保存 ---
    results = {"fold": FOLD, "size": SIZE, "n_test": len(test_ids), "landmarks": {}}
    all_vals = []
    for name, errs in all_errors.items():
        mre = sum(errs) / len(errs)
        sdr2 = sum(1 for e in errs if e <= 2.0) / len(errs) * 100
        sdr4 = sum(1 for e in errs if e <= 4.0) / len(errs) * 100
        results["landmarks"][name] = {"mre_mm": mre, "sdr2": sdr2, "sdr4": sdr4}
        all_vals.extend(errs)
    results["overall"] = {
        "mre_mm": sum(all_vals) / len(all_vals),
        "sdr2": sum(1 for e in all_vals if e <= 2.0) / len(all_vals) * 100,
        "sdr4": sum(1 for e in all_vals if e <= 4.0) / len(all_vals) * 100,
    }

    with open(out_path, "w") as f:
        json.dump(results, f, indent=2)
    print(f"MRE={results['overall']['mre_mm']:.2f}mm  SDR@4mm={results['overall']['sdr4']:.1f}%  → {out_path}")

print("\n全サイズ完了")